# 🔬 Lab 01: Multimodal Document Parsing & Ingestion Pipelines

---

## 🎯 Executive Overview & Production Challenge
In enterprise environments, proprietary knowledge rarely arrives as pristine plain text. Real enterprise repositories contain:
1. **Unstructured Technical Text:** Complex narrative manuals with dense engineering jargon.
2. **Semi-Structured Markdown & Tables:** Equipment operating envelopes, safety alarm thresholds, and maintenance intervals where row-column alignment is critical.
3. **Structured Logs & Telemetry:** JSON/CSV operational logs with timestamps, equipment IDs, and failure modes.
4. **Visual Diagrams & Technical Schematics:** Compressor performance maps, P&IDs, equipment cross-sections, and failure heatmaps.

### 🔴 The Naive RAG Failure Mode:
Standard text extractors (e.g. basic PDF text dumps) flatten tables into meaningless character soups and completely discard graphical charts. When an engineer asks: *"What is the surge flow limit at pressure ratio 5.6?"*, naive text RAG fails or hallucinates because the answer lives exclusively inside a visual curve.

### 🟢 The Production Architecture:
```
Heterogeneous Enterprise Sources
 ├── Technical Manual (.md) ──> Markdown Table Preservation Parser ─┐
 ├── Incident Telemetry (.json) ──> JSON Key-Value Structuring Parser ┼─> Unified Multi-Modal
 └── Equipment Curves (.png) ──> Local Vision LLM (Ollama/OpenAI) ──┘    Chunk Ingestion Index
```


In [2]:
# Pre-flight imports & verification
import os
import sys
import json
import base64
from pathlib import Path
from PIL import Image

# Ensure UTF-8 output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Robust sys.path injection so rag_universal_client is ALWAYS found
for candidate_dir in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs"),
    Path(__file__).parent if "__file__" in locals() else None,
]:
    if candidate_dir and candidate_dir.exists():
        resolved_str = str(candidate_dir.resolve())
        if resolved_str not in sys.path:
            sys.path.insert(0, resolved_str)

# Import Enterprise RAG Client
import rag_universal_client as rc

# Resilient path resolution for sample_documents (Prioritizing local lab folder)
DATA_DIR = rc.get_rag_data_dir()
print(f"[*] Verified Data Directory: {DATA_DIR}")

# Check files
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        print(f"  - Found: {f.name:32} ({f.stat().st_size:6d} bytes)")


[*] Verified Data Directory: C:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs\data\sample_documents
  - Found: compressor_curve.png             (106646 bytes)
  - Found: esg_regulatory_compliance.txt    ( 12094 bytes)
  - Found: incident_reports.json            ( 46185 bytes)
  - Found: scada_telemetry.csv              ( 33774 bytes)
  - Found: telemetry_monitor.py             (  3551 bytes)
  - Found: turbomachinery_manual.md         ( 18485 bytes)


---
## Step 1: Parsing Structured Operational Tables from Markdown Documents
Tabular data must retain its row-column relationship. We parse the operational manual and isolate the operating threshold table into a structured representation with embedded row metadata.


In [3]:
def extract_markdown_tables(filepath):
    """Extract and isolate markdown tables with context headers."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    lines = content.splitlines()
    tables = []
    current_table = []
    in_table = False
    section_header = "General"

    for line in lines:
        if line.startswith("## "):
            section_header = line.strip()
        if "|" in line:
            in_table = True
            current_table.append(line)
        else:
            if in_table and current_table:
                tables.append({
                    "section": section_header,
                    "table_raw": "\n".join(current_table),
                    "line_count": len(current_table)
                })
                current_table = []
                in_table = False

    if current_table:
        tables.append({
            "section": section_header,
            "table_raw": "\n".join(current_table),
            "line_count": len(current_table)
        })

    return tables

tables = extract_markdown_tables(DATA_DIR / "turbomachinery_manual.md")
print(f"[✓] Extracted {len(tables)} tables from Turbomachinery Manual.")
print("\n--- Sample Extracted Table with Section Context ---")
print(f"Section: {tables[0]['section']}")
print(tables[0]['table_raw'])


[✓] Extracted 5 tables from Turbomachinery Manual.

--- Sample Extracted Table with Section Context ---
Section: General
**Effective Date:** January 15, 2026 | **Review Cycle:** Annual  


---
## Step 2: Ingesting Semi-Structured JSON Telemetry & Incident Logs
JSON logs cannot simply be serialized as raw JSON strings because LLMs struggle with dense nested brackets. We transform JSON records into clean, descriptive natural language chunks while preserving key-value metadata for filtering.


In [4]:
def parse_incident_json(filepath):
    """Convert structured JSON incident reports into semantically searchable knowledge chunks."""
    with open(filepath, 'r', encoding='utf-8') as f:
        incidents = json.load(f)

    incident_chunks = []
    for inc in incidents:
        # Create a semantically dense textual representation
        text_repr = (
            f"Incident ID: {inc['incident_id']} at {inc['facility']} (Equipment: {inc['equipment_tag']}).\n"
            f"Subsystem: {inc['subsystem']} | Severity: {inc['severity']} | Downtime: {inc['downtime_hours']} hrs.\n"
            f"Event Summary: {inc['event_summary']}\n"
            f"Root Cause: {inc['root_cause']}\n"
            f"Corrective Actions:\n" + "\n".join([f"  - {act}" for act in inc['corrective_actions']])
        )
        
        chunk = {
            "chunk_id": f"INC_{inc['incident_id']}",
            "source_type": "structured_incident_telemetry",
            "metadata": {
                "incident_id": inc['incident_id'],
                "facility": inc['facility'],
                "equipment_tag": inc['equipment_tag'],
                "severity": inc['severity'],
                "downtime_hours": inc['downtime_hours'],
                "lead_engineer": inc['lead_engineer']
            },
            "content": text_repr
        }
        incident_chunks.append(chunk)

    return incident_chunks

incident_chunks = parse_incident_json(DATA_DIR / "incident_reports.json")
print(f"[✓] Converted {len(incident_chunks)} JSON incident logs into semantic chunks.")
print("\n--- Sample Incident Chunk ---")
print(incident_chunks[0]['content'])


[✓] Converted 25 JSON incident logs into semantic chunks.

--- Sample Incident Chunk ---
Incident ID: INC-2026-0814 at Fahud Central Processing Facility (Equipment: K-301A).
Subsystem: Dry Gas Seal (DGS) | Severity: CRITICAL | Downtime: 14.5 hrs.
Event Summary: Sudden primary vent leakage flow spike on compressor drive-end cartridge reaching 92.4 Nm3/hr, exceeding Level 2 ESD trip threshold of 90.0 Nm3/hr. Triple modular voting system commanded immediate automated train shutdown and blowdown to high-pressure flare via BDV-401.
Root Cause: Severe hydrocarbon liquid droplet condensation in primary seal gas buffer line caused by heating element failure on skid upstream scrubber coalescing filter. Liquid carryover breached rotating silicon carbide dynamic seal faces, causing micro-groove thermal chattering and face delamination.
Corrective Actions:
  - Replaced DE tandem dry gas seal cartridge with pre-calibrated spare unit (Serial #JC-28AT-9941).
  - Repaired failed 45 kW electric buffer 

---
## Step 3: Multimodal Vision Ingestion (Charts, Curves & Diagrams)
Technical manuals contain performance maps (e.g. compressor surge lines). Here, we demonstrate how to pass `compressor_curve.png` to a Vision-Language Model:
1. **Ollama Vision:** Local `llama3.2-vision` or `llava`.
2. **OpenAI Vision:** `gpt-4o-mini` if `OPENAI_API_KEY` is present.
3. **Structured Fallback:** Detailed synthetic OCR/VLM extraction if running in pure offline CPU mode.


In [5]:
import rag_universal_client as rc

def extract_image_intelligence(image_path):
    """
    Extract numerical thresholds, axes, and operational alerts from an engineering image.
    Uses Anthropic Claude Vision (Primary) with Local Ollama Vision backup.
    Strictly raises RuntimeError if all configured models are unreachable (Zero canned mocks).
    """
    print(f"[*] Analyzing engineering diagram: {Path(image_path).name} via Multi-Model Engine...")
    
    prompt = (
        "You are an expert turbomachinery diagnostic engineer at UST Global (UST). "
        "Analyze this centrifugal compressor performance curve diagram.\n"
        "Extract: 1) Diagram Title, 2) X and Y axes with units and operating ranges, 3) Critical speed curves, "
        "4) Specific Operating Points (e.g. Operating Point A), and 5) Surge Limit Line (SLL) safety margins."
    )
    
    # Real Model Invocation (Anthropic Claude Vision or Ollama Vision)
    res = rc.call_rag_vision(image_path=image_path, prompt=prompt)
    print(f"  [Model Used] : {res['tier_used']} (Latency: {res.get('latency_ms', 0)} ms)")
    return res["text"]

# Execute real vision extraction
image_path = DATA_DIR / "compressor_curve.png"
image_summary = extract_image_intelligence(image_path)
print("\n[✓] Multimodal Vision Extraction Output:")
print("=" * 80)
print(image_summary)
print("=" * 80)


[*] Analyzing engineering diagram: compressor_curve.png via Multi-Model Engine...
  [Model Used] : PRIMARY_VISION (claude-sonnet-4-6 - Live) (Latency: 26238.4 ms)

[✓] Multimodal Vision Extraction Output:
# Centrifugal Compressor Performance Map — Expert Diagnostic Analysis

---

## 1. 📋 Diagram Title

**"UST Global Solar Taurus 60 Centrifugal Compressor Performance Map"**

> This references a **Solar Turbines Taurus 60** gas turbine-driven centrifugal compressor, commonly deployed in **UST Global Enterprise Solutions** upstream gas compression facilities.

---

## 2. 📐 Axes — Units & Operating Ranges

| Axis | Parameter | Units | Range |
|------|-----------|-------|-------|
| **X-Axis** | Inlet Volumetric Flow Rate | m³/min | 20 → 100 |
| **Y-Axis** | Total Pressure Ratio (Pd/Ps) | Dimensionless | ~2.4 → 7.0 |

### Key Observations:
- **Pd** = Discharge (delivery) pressure
- **Ps** = Suction (inlet) pressure
- The ratio range of **2.4–7.0** indicates a **high-pressure ratio** single-s

---
## Step 4: Assembling Unified Multimodal Chunks
Now we construct a unified multimodal corpus where text, tables, and vision extractions coexist seamlessly with rich metadata tags.


In [6]:
def build_unified_corpus():
    corpus = []
    
    # 1. Text chunks from manual
    with open(DATA_DIR / "turbomachinery_manual.md", "r", encoding="utf-8") as f:
        text_content = f.read()
    
    sections = text_content.split("## ")
    for idx, sec in enumerate(sections[1:], 1):
        lines = sec.splitlines()
        header = lines[0].strip()
        body = "\n".join(lines[1:]).strip()
        corpus.append({
            "chunk_id": f"MANUAL_SEC_{idx}",
            "modality": "text",
            "source": "turbomachinery_manual.md",
            "section": header,
            "text": f"Section: {header}\n{body}"
        })

    # 2. Structured Incident Chunks
    for inc in incident_chunks:
        corpus.append({
            "chunk_id": inc["chunk_id"],
            "modality": "structured_log",
            "source": "incident_reports.json",
            "section": inc["metadata"]["facility"],
            "text": inc["content"]
        })

    # 3. Vision Extraction Chunk
    corpus.append({
        "chunk_id": "CHART_COMPRESSOR_CURVE_01",
        "modality": "vision_chart",
        "source": "compressor_curve.png",
        "section": "Performance Curve & Surge Limits",
        "text": image_summary
    })

    return corpus

unified_corpus = build_unified_corpus()
print(f"[✓] Unified Corpus Assembled with {len(unified_corpus)} heterogeneous chunks.")
for c in unified_corpus:
    print(f"  - [{c['modality'].upper():16}] ID: {c['chunk_id']:26} (Chars: {len(c['text'])})")


[✓] Unified Corpus Assembled with 55 heterogeneous chunks.
  - [TEXT            ] ID: MANUAL_SEC_1               (Chars: 66)
  - [TEXT            ] ID: MANUAL_SEC_2               (Chars: 793)
  - [TEXT            ] ID: MANUAL_SEC_3               (Chars: 1390)
  - [TEXT            ] ID: MANUAL_SEC_4               (Chars: 78)
  - [TEXT            ] ID: MANUAL_SEC_5               (Chars: 1202)
  - [TEXT            ] ID: MANUAL_SEC_6               (Chars: 489)
  - [TEXT            ] ID: MANUAL_SEC_7               (Chars: 84)
  - [TEXT            ] ID: MANUAL_SEC_8               (Chars: 1490)
  - [TEXT            ] ID: MANUAL_SEC_9               (Chars: 87)
  - [TEXT            ] ID: MANUAL_SEC_10              (Chars: 753)
  - [TEXT            ] ID: MANUAL_SEC_11              (Chars: 1336)
  - [TEXT            ] ID: MANUAL_SEC_12              (Chars: 75)
  - [TEXT            ] ID: MANUAL_SEC_13              (Chars: 660)
  - [TEXT            ] ID: MANUAL_SEC_14              (Chars: 937)
  - 

---
## Step 5: Querying across Text & Visual Chunks
Let's test whether our multimodal extraction answers queries that pure text RAG cannot resolve.
Query: *"What is the inlet flow rate at Operating Point A where compressor surge risk is identified?"*


In [7]:
query = "What is the inlet flow rate at Operating Point A where compressor surge risk is identified?"
print(f"User Query: '{query}'\n")

# Simple keyword & semantic overlap scan to verify multimodal retrieval
matching_chunks = []
query_terms = ["operating point a", "surge risk", "flow rate", "compressor"]

for c in unified_corpus:
    score = sum(1 for term in query_terms if term in c["text"].lower())
    if score > 0:
        matching_chunks.append((score, c))

matching_chunks.sort(key=lambda x: x[0], reverse=True)
best_chunk = matching_chunks[0][1]

print(f"[✓] Retrieved Top Modality Chunk: {best_chunk['chunk_id']} (Modality: {best_chunk['modality']})")
print(f"Source: {best_chunk['source']}")
print("\nRelevant Content Excerpt:")
for line in best_chunk['text'].splitlines():
    if any(k in line.lower() for k in ["operating point", "surge", "flow"]):
        print(f"  >>> {line}")


User Query: 'What is the inlet flow rate at Operating Point A where compressor surge risk is identified?'

[✓] Retrieved Top Modality Chunk: CHART_COMPRESSOR_CURVE_01 (Modality: vision_chart)
Source: compressor_curve.png

Relevant Content Excerpt:
  >>> | **X-Axis** | Inlet Volumetric Flow Rate | m³/min | 20 → 100 |
  >>> - Flow range spans **80 m³/min** bandwidth, covering both low-flow surge risk and high-flow choke territory
  >>> | Curve | Speed % | RPM | Color | Peak Pressure Ratio | Approx. Peak Flow |
  >>> - All three curves exhibit the **classic centrifugal compressor bell-shaped profile** — rising from surge margin at left, peaking, then falling toward choke/stonewall at right
  >>> - **100% speed curve** is most aggressive — highest head capability but **narrowest stable operating window** relative to surge
  >>> ## 4. 🔴 Specific Operating Points
  >>> ### Operating Point A — **SURGE RISK FLAGGED**
  >>> | **Flow Rate** | 38 m³/min |
  >>> | **Risk Status** | ⚠️ **SURGE RISK

---
## 🏁 Production Engineering Takeaways & Exercises
1. **Never rely on naive text extractors** for technical enterprise assets. Use markdown serialization for tables and OCR/VLM extraction for charts.
2. **Tag every chunk with `modality` and `source_type`** in the metadata payload to enable pre-filtering and source attribution.
3. **Try It Yourself Exercise:** Modify `extract_markdown_tables` to output JSON objects where table column headers become dictionary keys for each row.
